# tinydistill: deploy a distilled model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andreas-Hum/tinydistill/blob/main/notebooks/deploy.ipynb)

Ways to use a distilled model once it exists: from Python, behind an HTTP server, in a web page that runs it
in the visitor's browser, and from the Hugging Face Hub. The first cell distils a small model so the notebook
stands on its own; with a model of your own, point `MODEL` at its `onnx` folder instead.

A GPU runtime makes the first cell faster (Runtime > Change runtime type > T4 GPU).

In [ ]:
!pip install -q "tinydistill[hf,plot]"

Downloads from the Hugging Face Hub work without an account. If they are slow, or you see a warning about
unauthenticated requests, add a token: in Colab, open the key icon in the left sidebar and add a secret named
`HF_TOKEN` with a token from https://huggingface.co/settings/tokens (read access is enough).

In [ ]:
!curl -sLO https://raw.githubusercontent.com/Andreas-Hum/tinydistill/main/examples/financial-tweets/task.yaml
!tinydistill run task.yaml --data hf:zeroshot/twitter-financial-news-sentiment:train --limit 1000 \
  --test hf:zeroshot/twitter-financial-news-sentiment:validation --test-limit 300 \
  --teacher Qwen/Qwen2.5-1.5B-Instruct --int8 --out run
MODEL = "run/onnx"

## 1. From Python

`Predictor` loads the int8 model with onnxruntime. Batches are much faster per text than one text at a time.

In [ ]:
import time

from tinydistill import Predictor

clf = Predictor(MODEL)
texts = ["Shares jump after strong quarterly results"] * 256
clf.predict(texts[:8])  # warm up
for batch in (1, 32):
    t0 = time.perf_counter()
    clf.predict(texts, batch_size=batch)
    print(f"batch {batch:>2}: {1000 * (time.perf_counter() - t0) / len(texts):.2f} ms per text")
print(clf.describe(["Regulators fine the bank for misleading customers"]))

## 2. An HTTP server

`tinydistill serve` answers `POST /predict` with JSON. It uses only the Python standard library; put it behind
your usual web server for production.

In [ ]:
import json
import subprocess
import urllib.request

server = subprocess.Popen(["tinydistill", "serve", MODEL, "--port", "8000"])
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health")
        break
    except OSError:
        time.sleep(1)
request = urllib.request.Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps({"texts": ["Oil prices fall as demand weakens"]}).encode(),
    headers={"Content-Type": "application/json"},
)
print(json.loads(urllib.request.urlopen(request).read()))
server.terminate()

## 3. A web page that runs the model in the browser

`tinydistill web` writes a static folder: the page, plus the model in the layout transformers.js expects. Any
static host works (GitHub Pages, for example). The page also shows what happens inside the model:
[live example](https://andreas-hum.github.io/tinydistill/).

In [ ]:
!tinydistill web {MODEL} --out site --title "Financial tweet sentiment"
!cd site && zip -qr ../site.zip . && ls -la ..//site.zip

## 4. The Hugging Face Hub

Uploading needs a token with **write** access (https://huggingface.co/settings/tokens). After the upload,
anyone can use the model with `Predictor("hf:user/name")` or `tinydistill predict hf:user/name "text"`.
Set `REPO` and run the cell; it is skipped while `REPO` is empty.

In [ ]:
REPO = ""  # for example "your-name/financial-tweets-minilm"

if REPO:
    from huggingface_hub import notebook_login

    notebook_login()
    !tinydistill push {MODEL} --repo {REPO}
    print(Predictor(f"hf:{REPO}").predict(["Shares rally after the merger is approved"]))